# Thêm hai seed cho mô hình từ điển và BiLSTM (chạy ngầm trên Kaggle, chỉ dùng CPU)

Notebook này huấn luyện 4 lần chạy để cả sáu mô hình của Chương 4 đều có 5 seed (42, 1337, 2024, 7, 123):

| Mô hình | Seed mới | Ba seed cũ chạy ở đâu |
|---|---|---|
| `lexicon` (từ điển cảm xúc) | 7, 123 | CPU máy học viên |
| `bilstm` (BiLSTM với attention) | 7, 123 | CPU máy học viên |

**Chỉ dùng CPU**, vì ba seed cũ của hai mô hình này đều huấn luyện trên CPU (Bảng 4.6). Notebook ép mọi lần chạy về CPU kể
cả khi lỡ bật GPU, nên hãy để *Accelerator* là *None* để không tốn giờ GPU. Thời gian ước tính 2 đến 4 giờ: mô hình từ
điển vài phút một seed, BiLSTM khoảng 55 phút một seed trên máy học viên, CPU của Kaggle có thể chậm hơn. Notebook tự dừng
nhận việc mới ở mốc 11 giờ.

## Cách chạy (một lần bấm, đóng trình duyệt được)

1. **Input** (khung bên phải): gắn **dataset đã dùng ở hai lần trước**, có `visfd_train.jsonl`, `visfd_dev.jsonl`,
   `visfd_test.jsonl`. BiLSTM dựng bảng từ vựng từ tập huấn luyện, nên cần đủ ba file.
2. **Settings**: *Accelerator* = **None**; *Internet* = **On** (kéo mã từ GitHub và tải bộ tách từ PhoBERT).
3. Góc trên bên phải: **Save Version** → **Save & Run All (Commit)** → **Save**.
4. Khoảng 10 phút sau, mở trang version → tab *Logs*: phải thấy `bat dau lexicon/seed7`, rồi
   `lexicon/seed7: xong`. Thấy rồi thì đóng trình duyệt.

**Chạy nốt lần sau** (chỉ khi có việc "bỏ qua: hết giờ" hoặc "lỗi"): *Add Input* → *Your Work* → chọn **output của
version trước** của chính notebook này, rồi Save Version lại. Việc nào đã có kết quả sẽ được chép sang và bỏ qua.

## 1. Mốc thời gian và kiểm phần cứng

In [ ]:
# Moc cua CA phien: hang doi o muc 6 dung moc nay de khong vuot gioi han 12 gio cua Kaggle.
import os, time, torch

BAT_DAU_PHIEN = time.time()
SO_NHAN = os.cpu_count()
print(f"CPU: {SO_NHAN} nhan | torch {torch.__version__}")
if torch.cuda.is_available():
    print(f"CHU Y: dang bat GPU ({torch.cuda.get_device_name(0)}). Notebook van ep chay CPU de giong ba seed cu; "
          "lan sau dat Settings > Accelerator = None de khong ton gio GPU.")
else:
    print("OK: khong co GPU, chay CPU nhu ba seed cu.")

## 2. Lấy mã từ GitHub

In [ ]:
import subprocess
from pathlib import Path

# Ma nguon va checkpoint de o /tmp, KHONG o /kaggle/working: moi thu trong /kaggle/working thanh output cua version.
# Ten thu muc khong dat la "nsmgat": trung ten goi Python, import se nham sang thu muc nay.
REPO = Path("/tmp/nsmgat-repo")
URL = "https://github.com/2611326-NguyenMinhTrong/nsmgat-vietnamese-absa.git"

if not REPO.exists():
    r = subprocess.run(["git", "clone", "--quiet", URL, str(REPO)], capture_output=True, text=True)
    assert r.returncode == 0, "KHONG KEO DUOC MA. Bat Settings > Internet = On.\n" + r.stderr[-500:]
COMMIT = subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%H %s"],
                        capture_output=True, text=True).stdout.strip()
print(COMMIT)
for f in ("configs/lexicon.yaml", "configs/bilstm.yaml", "data/lexicon_from_train.json"):
    assert (REPO / f).exists(), f"MA TREN GITHUB THIEU {f}."

## 3. Cài mã nguồn

Không cài `requirements.txt`: file đó có `torch`, cài đè lên bản torch sẵn có của Kaggle.

In [ ]:
%pip install -q transformers pyyaml
%pip install -q -e /tmp/nsmgat-repo
!python -c "import nsmgat, transformers, torch; print('nsmgat:', nsmgat.__file__); print('transformers', transformers.__version__, '| torch', torch.__version__)"

## 4. Nối dữ liệu, tải bộ tách từ PhoBERT một lần

Ba file dữ liệu được kiểm số dòng và mã băm SHA-256, để chắc đó đúng là dữ liệu đã cho các con số của báo cáo.
BiLSTM chỉ dùng bộ tách từ của PhoBERT (bảng nhúng của nó học từ đầu), không dùng trọng số PhoBERT.

In [ ]:
import hashlib, os, subprocess
from pathlib import Path

GOC_VAO = Path("/kaggle/input")
# So dong va SHA-256 cua ba file o may hoc vien (do ngay 16/10/2026)
DU_LIEU = {
    "visfd_train.jsonl": (23872, "674db57d5a4969cb1cb1f0ea4757e94c9762d57a1e7d81488d4ab65971074790"),
    "visfd_dev.jsonl": (3316, "208653f1f02c423b0627036b2575716c8b3d6a859b10a69b11227c3204487060"),
    "visfd_test.jsonl": (6722, "f6d714f298b02476bb47adeb52ce6daf198d60f73cf8f7964e2253ebfea57a5e"),
}

for ten, (so_dong, bam) in DU_LIEU.items():
    ds = sorted(p for p in GOC_VAO.rglob(ten) if p.is_file())
    assert len(ds) == 1, f"CAN DUNG MOT FILE {ten} TRONG INPUT, DANG CO {len(ds)}: {[str(p) for p in ds]}"
    p = ds[0]
    dem = sum(1 for _ in p.open(encoding="utf-8"))
    sha = hashlib.sha256(p.read_bytes()).hexdigest()
    assert dem == so_dong and sha == bam, f"{ten} KHAC FILE O MAY: {dem} dong (can {so_dong}), sha256 {sha[:16]} (can {bam[:16]})."
    dich = REPO / "data/processed" / ten
    dich.parent.mkdir(parents=True, exist_ok=True)
    if dich.is_symlink() or dich.exists():
        dich.unlink()
    os.symlink(p, dich)
    print(f"{ten:20} {dem:>7,} dong, khop file o may  <- {p}")

r = subprocess.run(["python", "-c", "from transformers import AutoTokenizer; "
                    "AutoTokenizer.from_pretrained('vinai/phobert-base-v2'); print('Da tai bo tach tu PhoBERT')"],
                   capture_output=True, text=True)
assert r.returncode == 0, "KHONG TAI DUOC BO TACH TU PhoBERT:\n" + r.stderr[-800:]
print(r.stdout.strip())

## 5. Danh sách việc

Chạy lần lượt từng việc, mỗi việc dùng hết số nhân CPU. Phút ước tính: từ điển lấy từ Bảng 4.6; BiLSTM lấy gần gấp đôi số
đo ở máy học viên (53 phút) vì CPU của Kaggle chưa đo.

In [ ]:
import json, shutil
from pathlib import Path

RA = Path("/kaggle/working/ket_qua")           # metrics, du doan, nhat ky -> dong goi thanh ket_qua.zip
GIU = Path("/kaggle/working/checkpoints_giu")  # best.pt cua ca bon lan chay (nho: tu dien 9 tham so, BiLSTM 4 trieu)
RA.mkdir(parents=True, exist_ok=True)

SEED_MOI = (7, 123)
# (ten thi nghiem = --exp-name, config, --model, seed, phut uoc tinh)
VIEC = (
    [("lexicon", "configs/lexicon.yaml", "lexicon", s, 6) for s in SEED_MOI]
    + [("bilstm", "configs/bilstm.yaml", "bilstm", s, 100) for s in SEED_MOI]
)
SO_THAM_SO = {"lexicon": 9, "bilstm": 4017412}  # phai bang ba seed cu

# Chay not: neu output cua version truoc duoc gan lam input, viec nao da co metrics.json thi chep sang va bo qua.
da_co = set()
for m in sorted(GOC_VAO.rglob("ket_qua/*/seed*/metrics.json")):
    exp, seed = m.parent.parent.name, int(m.parent.name[4:])
    dich = RA / exp / f"seed{seed}"
    if not (dich / "metrics.json").exists():
        shutil.copytree(m.parent, dich, dirs_exist_ok=True)
    da_co.add((exp, seed))
CON_LAI = [v for v in VIEC if (v[0], v[3]) not in da_co]

print(f"{len(VIEC)} viec; {len(VIEC) - len(CON_LAI)} viec da co tu version truoc; con {len(CON_LAI)}")
print(f"Uoc tinh {sum(v[4] for v in CON_LAI) / 60:.1f} gio")
for v in CON_LAI:
    print(f"  {v[0]:8} seed {v[3]:<5} ~{v[4]} phut")

## 6. Chạy hàng đợi

Mỗi việc là một lệnh `python -m nsmgat.train` riêng, với `CUDA_VISIBLE_DEVICES` rỗng để chắc chắn chạy CPU. Nhật ký ghi
vào `ket_qua/_nhat_ky/`. Một việc chỉ được bắt đầu khi thời gian còn lại tới mốc 11 giờ ít nhất bằng 1,3 lần ước tính;
việc chạy quá hạn bị dừng và ghi lỗi.

In [ ]:
import json, os, shutil, subprocess, time

GIOI_HAN_GIO = 11.0   # Kaggle cat phien o 12 gio; chua mot gio de dong goi
HE_SO_AN_TOAN = 1.3
HAN_CHOT = BAT_DAU_PHIEN + GIOI_HAN_GIO * 3600
NHAT_KY = RA / "_nhat_ky"
NHAT_KY.mkdir(exist_ok=True)
TRANG_THAI = RA / "trang_thai.json"
bang = json.loads(TRANG_THAI.read_text(encoding="utf-8")) if TRANG_THAI.exists() else []


def ghi(muc):
    bang.append(muc)
    TRANG_THAI.write_text(json.dumps(bang, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"{muc['exp']}/seed{muc['seed']}: {muc['trang_thai']}"
          + (f" sau {muc['phut']} phut" if "phut" in muc else ""), flush=True)


for exp, cfg, model, seed, uoc in CON_LAI:
    muc = {"exp": exp, "seed": seed, "thiet_bi": "cpu", "uoc_tinh_phut": uoc}
    con = HAN_CHOT - time.time()
    if con < uoc * 60 * HE_SO_AN_TOAN:
        muc["trang_thai"] = f"bo qua: het gio (con {con / 60:.0f} phut)"
        ghi(muc)
        continue
    print(f"bat dau {exp}/seed{seed} (~{uoc} phut)", flush=True)
    lenh = ["python", "-m", "nsmgat.train", "--config", cfg, "--model", model, "--exp-name", exp, "--seed", str(seed)]
    moi_truong = dict(os.environ, CUDA_VISIBLE_DEVICES="", PYTHONUNBUFFERED="1")
    t0 = time.time()
    with open(NHAT_KY / f"{exp}_seed{seed}.txt", "w", encoding="utf-8") as f:
        try:
            rc = subprocess.run(lenh, cwd=REPO, env=moi_truong, stdout=f, stderr=subprocess.STDOUT,
                                timeout=max(60, con - 300)).returncode
        except subprocess.TimeoutExpired:
            rc = "qua han"
    muc["phut"] = round((time.time() - t0) / 60, 1)

    ket = REPO / "results" / exp / f"seed{seed}"
    log = REPO / "logs" / exp / f"seed{seed}.log"
    if rc == 0 and (ket / "metrics.json").exists():
        dich = RA / exp / f"seed{seed}"
        dich.mkdir(parents=True, exist_ok=True)
        for ten in ("metrics.json", "predictions.jsonl"):
            shutil.copy(ket / ten, dich / ten)
        shutil.copy(log, dich / "train.log")
        m = json.loads((ket / "metrics.json").read_text(encoding="utf-8"))
        muc.update(trang_thai="xong", macro_f1=m["test"]["macro_f1"], accuracy=m["test"]["accuracy"],
                   n_params=m["n_params"], config_hash=m["config_hash"],
                   chay_cpu="device=cpu" in log.read_text(encoding="utf-8"))
    else:
        muc["trang_thai"] = f"loi (ma thoat {rc}), xem _nhat_ky/{exp}_seed{seed}.txt"

    ck = REPO / "checkpoints" / exp / f"seed{seed}"
    if muc["trang_thai"] == "xong" and (ck / "best.pt").exists():
        (GIU / exp / f"seed{seed}").mkdir(parents=True, exist_ok=True)
        shutil.copy(ck / "best.pt", GIU / exp / f"seed{seed}" / "best.pt")
    shutil.rmtree(ck, ignore_errors=True)
    ghi(muc)

print(f"HET HANG DOI sau {(time.time() - BAT_DAU_PHIEN) / 3600:.1f} gio")

## 7. Tổng hợp, tự kiểm và đóng gói

Bảng dưới gộp seed mới với ba seed cũ có sẵn trong repo, chỉ để xem nhanh. Kiểm định làm ở máy sau khi mang kết quả về.

In [ ]:
import json, platform, shutil, statistics, time, torch, transformers
from pathlib import Path

bang = json.loads(TRANG_THAI.read_text(encoding="utf-8")) if TRANG_THAI.exists() else []
print(f"{'thi nghiem':10} {'seed':>5}  {'macro-F1':>8} {'acc':>7} {'phut':>6}  trang thai")
for m in bang:
    f1 = f"{m['macro_f1'] * 100:8.2f}" if "macro_f1" in m else " " * 8
    acc = f"{m['accuracy'] * 100:7.2f}" if "accuracy" in m else " " * 7
    print(f"{m['exp']:10} {m['seed']:>5}  {f1} {acc} {m.get('phut', ''):>6}  {m['trang_thai']}")

print("\nMacro-F1 gop ca seed cu (tu repo) va seed moi:")
for exp in ("lexicon", "bilstm"):
    gia_tri = {}
    for goc in (REPO / "results", RA):
        for p in sorted((goc / exp).glob("seed*/metrics.json")):
            gia_tri[p.parent.name] = json.loads(p.read_text(encoding="utf-8"))["test"]["macro_f1"] * 100
    if gia_tri:
        v = list(gia_tri.values())
        sd = f" +/- {statistics.stdev(v):.2f}" if len(v) > 1 else ""
        print(f"  {exp:8} {len(v)} seed: {statistics.mean(v):.2f}{sd}   ({', '.join(f'{k[4:]}={x:.2f}' for k, x in gia_tri.items())})")

print()
for m in bang:
    if m["trang_thai"] != "xong":
        continue
    dung_ts = m["n_params"] == SO_THAM_SO[m["exp"]]
    print(("OK   " if dung_ts and m["chay_cpu"] else "SAI  ")
          + f"{m['exp']}/seed{m['seed']}: so tham so {m['n_params']:,} (can {SO_THAM_SO[m['exp']]:,}), "
          + ("chay CPU" if m["chay_cpu"] else "KHONG chay CPU"))

moi_truong = {"commit": COMMIT, "thiet_bi": "cpu", "so_nhan_cpu": SO_NHAN, "torch": torch.__version__,
              "transformers": transformers.__version__, "python": platform.python_version(),
              "so_gio_phien": round((time.time() - BAT_DAU_PHIEN) / 3600, 2),
              "ghi_chu": "CPU cua Kaggle khac CPU may hoc vien nen thoi gian huan luyen KHONG so duoc voi Bang 4.6"}
(RA / "moi_truong.json").write_text(json.dumps(moi_truong, ensure_ascii=False, indent=1), encoding="utf-8")
shutil.make_archive("/kaggle/working/ket_qua", "zip", "/kaggle/working", "ket_qua")
print("\nDa dong goi /kaggle/working/ket_qua.zip")

## 8. Mang về máy

1. Mở trang version vừa chạy → tab *Output* → tải **`ket_qua.zip`** (vài MB). Thư mục `checkpoints_giu/` (khoảng 32 MB)
   không bắt buộc.
2. Đặt `ket_qua.zip` vào `data\cache\kaggle_them_seed\` của repo (thư mục này không đẩy lên GitHub). Không giải nén,
   không đổi tên.
3. Báo Claude, kèm các dòng `OK` hoặc `SAI` ở cuối ô 7. Claude kiểm từng file rồi mới đưa vào `results/`.

Không cần tải notebook về: Kaggle giữ bản chạy kèm nhật ký trong trang version.